# Coupling matrices along Sinkhorn iterations

This notebook generates `fig:sinkhorn-coupling-iterations`.  Instead of looking only at marginal errors or dual potentials, it displays the current coupling matrix
$$
P^{(k)}=\operatorname{diag}(u^{(k)})K\operatorname{diag}(v^{(k)})
$$
at selected iterations.  The color scale is shared across panels so that the mass concentration and stabilization of the coupling can be compared directly.

In [1]:

from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot

from figure_style import BLUE, RED, VIOLET, GRAY, LIGHT_GRAY, ORANGE, box_axes, figure_dir, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()
np.random.seed(0)

def normal_pdf(x, mean, std):
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))


def mixture_pdf(x, weights, means, stds):
    out = np.zeros_like(x, dtype=float)
    for w, m, s in zip(weights, means, stds):
        out += w * normal_pdf(x, m, s)
    return out


def sinkhorn_histograms(n=90):
    grid = np.linspace(-3.25, 3.15, n)
    alpha_density = mixture_pdf(grid, [0.58, 0.42], [-1.95, -0.10], [0.34, 0.54])
    beta_density = mixture_pdf(grid, [0.42, 0.58], [-0.75, 1.55], [0.42, 0.36])
    a = alpha_density / alpha_density.sum()
    b = beta_density / beta_density.sum()
    C = (grid[:, None] - grid[None, :]) ** 2
    C = C / np.median(C[C > 0])
    return grid, alpha_density, beta_density, a, b, C


def scaling_states(K, a, b, n_full=40):
    P = K / K.sum()
    out = [('initial', P.copy())]
    for k in range(1, n_full + 1):
        P = (a / np.maximum(P.sum(axis=1), 1e-300))[:, None] * P
        out.append((f'row-{k}', P.copy()))
        P = P * (b / np.maximum(P.sum(axis=0), 1e-300))[None, :]
        out.append((f'column-{k}', P.copy()))
    return out


def sinkhorn_error_curves(a, b, C, epsilons, n_iter=130):
    curves = []
    for eps in epsilons:
        K = np.exp(-C / eps)
        P = K / K.sum()
        errs = [0.5 * (np.abs(P.sum(axis=1)-a).sum() + np.abs(P.sum(axis=0)-b).sum())]
        for _ in range(n_iter):
            P = (a / np.maximum(P.sum(axis=1), 1e-300))[:, None] * P
            errs.append(0.5 * (np.abs(P.sum(axis=1)-a).sum() + np.abs(P.sum(axis=0)-b).sum()))
            P = P * (b / np.maximum(P.sum(axis=0), 1e-300))[None, :]
            errs.append(0.5 * (np.abs(P.sum(axis=1)-a).sum() + np.abs(P.sum(axis=0)-b).sum()))
        curves.append((eps, np.asarray(errs)))
    return curves


def matrix_with_marginal_curves(P, a, b, path, *, gamma=0.45, show_current=True, compact=False, vmax=None):
    n, m = P.shape
    fig, ax = plt.subplots(figsize=(2.18, 2.04) if not compact else (2.05, 1.95))
    scale = max(float(P.max() if vmax is None else vmax), 1e-15)
    image = np.clip(P / scale, 0, 1)
    ax.imshow(image ** gamma, cmap='Greys', origin='lower', extent=(-0.5, m-0.5, -0.5, n-0.5), interpolation='nearest', vmin=0, vmax=1, aspect='auto')

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.72)
        spine.set_color('#2b2b2b')

    row = P.sum(axis=1)
    col = P.sum(axis=0)
    left_base = -0.5 - 0.035 * m
    top_base = n - 0.5 + 0.035 * n
    left_scale = 0.13 * m / max(a.max(), row.max(), 1e-15)
    top_scale = 0.13 * n / max(b.max(), col.max(), 1e-15)
    yy = np.arange(n)
    xx = np.arange(m)

    ax.plot(left_base - left_scale * a, yy, color=RED, lw=0.95, zorder=4)
    ax.fill_betweenx(yy, left_base, left_base - left_scale * a, color=RED, alpha=0.12, linewidth=0, zorder=3)
    ax.plot(xx, top_base + top_scale * b, color=BLUE, lw=0.95, zorder=4)
    ax.fill_between(xx, top_base, top_base + top_scale * b, color=BLUE, alpha=0.12, linewidth=0, zorder=3)

    if show_current:
        ax.plot(left_base - left_scale * row, yy, color=VIOLET, lw=0.80, alpha=0.90, zorder=5)
        ax.fill_betweenx(yy, left_base, left_base - left_scale * row, color=VIOLET, alpha=0.12, linewidth=0, zorder=4)
        ax.plot(xx, top_base + top_scale * col, color=VIOLET, lw=0.80, alpha=0.90, zorder=5)
        ax.fill_between(xx, top_base, top_base + top_scale * col, color=VIOLET, alpha=0.12, linewidth=0, zorder=4)

    ax.set_xlim(-0.5 - 0.18 * m, m - 0.5)
    ax.set_ylim(-0.5, n - 0.5 + 0.18 * n)
    ax.set_xticks([])
    ax.set_yticks([])
    save_pdf(fig, path, pad_inches=0.04)
    plt.close(fig)

fig_name = "sinkhorn-coupling-iterations"
out = figure_dir(fig_name)


## Same setting as the dual-potential figures

The problem is the same one-dimensional Gaussian-mixture setting used in `fig:sinkhorn-dual-potentials-epsilon`, with fixed regularization $\varepsilon=0.045$.  We export matrices after complete row-column scaling cycles.

In [2]:

grid, alpha_density, beta_density, a, b, C = sinkhorn_histograms(n=86)
epsilon = 0.045
K = np.exp(-C / epsilon)
P = K / K.sum()
cycles = {0: P.copy()}
for k in range(1, 61):
    P = (a / np.maximum(P.sum(axis=1), 1e-300))[:, None] * P
    P = P * (b / np.maximum(P.sum(axis=0), 1e-300))[None, :]
    if k in {1, 3, 12, 60}:
        cycles[k] = P.copy()

# The final maximum defines the common visual scale.
common_max = max(v.max() for v in cycles.values())
for k, filename in [(0, 'iter-0.pdf'), (1, 'iter-1.pdf'), (3, 'iter-3.pdf'), (12, 'iter-12.pdf'), (60, 'iter-60.pdf')]:
    Pk = cycles[k]
    # Use the same scale by multiplying with a harmless common normalization.
    matrix_with_marginal_curves(Pk, a, b, out / filename, gamma=0.42, show_current=False, compact=True, vmax=common_max)
